# 📝 ملخّص النصوص العربية باستخدام Claude AI
## Arabic Text Summarizer — Anthropic Claude API

هذا النوتبوك يستخدم **Anthropic Claude API** لتلخيص النصوص العربية.

---

## 1. تثبيت المكتبات المطلوبة

In [24]:
!pip install transformers datasets evaluate rouge-score pandas scikit-learn --quiet

In [25]:
import pandas as pd
import re
import evaluate
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [26]:
!kaggle datasets download -d abdalrahmanshahrour/arabicsummarization -p /content/data --unzip

file_path = '/content/data/data.csv'
df = pd.read_csv(file_path)

print(f"✅ عدد السجلات: {df.shape[0]}")

Dataset URL: https://www.kaggle.com/datasets/abdalrahmanshahrour/arabicsummarization
License(s): unknown
100% 6.30M/6.30M [00:00<00:00, 91.9MB/s]

✅ عدد السجلات: 5227


In [27]:
def clean_arabic_text(text):
    if pd.isna(text):
        return ""
    text = str(text)

    # إزالة التشكيل
    text = re.sub(r'[\u064B-\u0652]', '', text)

    # توحيد الحروف
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)

    # إزالة الرموز
    text = re.sub(r'[^\w\s]', '', text)

    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['clean_text'] = df['Processed Text'].apply(clean_arabic_text)
df['clean_summary'] = df['summarizer'].apply(clean_arabic_text)

# حذف البيانات الفارغة
df = df.dropna(subset=['clean_text', 'clean_summary'])
df = df[df['clean_text'].str.len() > 20]

print(f"✅ بعد التنظيف: {len(df)}")

✅ بعد التنظيف: 5227


In [28]:
SAMPLE_SIZE = 200  # زوديها لو الجهاز يستحمل

if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 160, Test: 40


In [29]:
model_name = "csebuetnlp/mT5_multilingual_XLSum"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [30]:
generated_summaries = []

print("🚀 جاري التلخيص...")

for _, row in test_df.iterrows():
    text = row['clean_text']
    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=20,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    generated_summaries.append(summary)

test_df['generated_summary'] = generated_summaries

🚀 جاري التلخيص...


In [31]:
def normalize_arabic(text):
    text = text.lower()
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

test_df['gen_clean'] = test_df['generated_summary'].apply(normalize_arabic)
test_df['ref_clean'] = test_df['clean_summary'].apply(normalize_arabic)

In [36]:
!pip install bert-score --quiet

from bert_score import score

preds = test_df['generated_summary'].tolist()
refs = test_df['clean_summary'].tolist()

P, R, F1 = score(preds, refs, lang="ar")

print(f"\n📊 BERTScore:")
print(f"Precision: {P.mean():.4f}")
print(f"Recall: {R.mean():.4f}")
print(f"F1: {F1.mean():.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 BERTScore:
Precision: 0.7756
Recall: 0.7582
F1: 0.7661


In [33]:
print("\n--- أمثلة ---")

for i, row in test_df.head(3).iterrows():
    print("\n📄 النص:")
    print(row['clean_text'][:150])

    print("\n🎯 المرجعي:")
    print(row['clean_summary'])

    print("\n🤖 المولد:")
    print(row['generated_summary'])

    print("-" * 50)


--- أمثلة ---

📄 النص:
اكد ناجح المثلوثي طبيب عام يعمل بمدينه بوفيشه ولايه سوسه تسجيل حالات اصابه بالتهاب الكبد الفيروسي صنف بالجهه واضاف تصريح الجوهره ام انه الاشتباه عدد ا

🎯 المرجعي:
اكد ناجح المثلوثي طبيب عام يعمل بمدينه بوفيشه من ولايه سوسه تسجيل 7 حالات اصابه بالتهاب الكبد الفيروسي صنف ا بالجهه

🤖 المولد:
قال ناجح المثلوثي طبيب عام يعمل بمدينه بوفيشه ولايه سوسه انه تم تسجيل حالات اصابه بالتهاب الكبد الفيروسي صنف بالجهه واصابة ثلاثة اشخاص بينهم رضيع يبلغ العمر حوالي سنوات.
--------------------------------------------------

📄 النص:
نشر يعرف ب قوه الردع الخاصه الليبيه فيديو لعنصر تونسي ينتمي لتنظيم داعش الارهابي يدعي محمد بن محسن الغربي ب زيد واعترف الغربي الفيديو بالتخطيط رفقه عن

🎯 المرجعي:
نشر ما يعرف بـقوه الردع الخاصه الليبيه فيديو لعنصر تونسي ينتمي لتنظيم داعش الارهابي يدعي محمد بن محسن الغربي المكني بـابو زيد واعترف الغربي في هذا الفيديو بالتخطيط رفقه عناصر ارهابيه اخري تنشط في صبراته اغلبهم من التونسيين بالاعداد لتنفيذ عمليات ارهابيه في تونس وصبراته باستخدام سيارات مفخخه



In [35]:
def compression_ratio(original, summary):
    return len(original.split()) / max(len(summary.split()), 1)

test_df['compression'] = test_df.apply(
    lambda row: compression_ratio(row['clean_text'], row['generated_summary']),
    axis=1
)

print(f"\n🗜️ Avg Compression: {test_df['compression'].mean():.2f}x")



🗜️ Avg Compression: 2.96x
